# Whisper Small Fine-Tuning for Nepali Speech Recognition

This notebook records the full fine-tuning experiment for Nepali speech recognition using **Whisper Small**.

The model was fine-tuned on the training split and then evaluated on the validation and test splits. The final test results are also compared with the pretrained Whisper Small model.

A few practical notes about this run:

- The experiment was run in Google Colab on a Tesla T4 GPU.
- Full fine-tuning took a long time because all model parameters were being trained and the available GPU resources were limited.
- Checkpoints were saved during training so the run could continue after a Colab session ended.
- The outputs already saved in this notebook are kept. The notebook has not been rerun, so the displayed results remain the results from the original experiment.

The main experiment is arranged first. Earlier setup attempts and checkpoint recovery cells are kept near the end as notes from the process.

## 1. Connect Google Drive and check the project

The dataset, audio files, checkpoints, and result files are stored in Google Drive. The first cell mounts Drive so the notebook can access them.

The project check also loads the train, validation, and test manifests and confirms that the audio folders are available.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import os
import pandas as pd
import torch
from pathlib import Path

# Project directory
PROJECT_DIR = Path("/content/drive/MyDrive/Nepali_Whisper_Project")

# Manifest paths
TRAIN_CSV = PROJECT_DIR / "train_manifest.csv"
VAL_CSV = PROJECT_DIR / "validation_manifest.csv"
TEST_CSV = PROJECT_DIR / "test_manifest.csv"

# Audio directories
TRAIN_AUDIO_DIR = PROJECT_DIR / "audio" / "train"
VAL_AUDIO_DIR = PROJECT_DIR / "audio" / "validation"
TEST_AUDIO_DIR = PROJECT_DIR / "audio" / "test"

# Check GPU
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

# Check project files
print("\nProject directory exists:", PROJECT_DIR.exists())
print("Project files:", os.listdir(PROJECT_DIR))

# Load manifests
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("\n===== DATASET SUMMARY =====")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

# Check audio directories
for name, path in [
    ("Train", TRAIN_AUDIO_DIR),
    ("Validation", VAL_AUDIO_DIR),
    ("Test", TEST_AUDIO_DIR)
]:
    print(f"{name} audio directory exists:", path.exists())

print("\nTraining columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())


PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Project directory exists: True
Project files: ['metadata.csv', 'audio_manifest.csv', 'duration_sample.csv', 'duration_statistics.csv', 'speaker_statistics.csv', 'speaker_assignment.csv', 'train_manifest.csv', 'validation_manifest.csv', 'test_manifest.csv', 'selected_manifest.csv', 'audio', 'audio_quality_report_backup.csv', 'audio_quality_report.csv', 'transcription_quality_report.csv', 'transcription_quality_report_backup.csv', 'download_progress.csv', 'drive_sync_test.txt', 'results', 'whisper_small_fft']

===== DATASET SUMMARY =====
Training samples: 34941
Validation samples: 7622
Test samples: 7495
Train audio directory exists: True
Validation audio directory exists: True
Test audio directory exists: True

Training columns: ['file_id', 'archive_prefix', 'archive_path', 'file_size', 'compressed_size', 'speaker_id', 'transcription', 'split']
Validation columns: ['file_id', 'archive_prefix', 'archive

## 2. Load the dataset and Whisper processor

The manifest files contain the audio file IDs and their transcripts. Each split is loaded separately so that training, validation, and test data remain separate.

The Whisper processor prepares audio for the model and converts transcripts into tokens. The language and task are set to Nepali transcription.

In [5]:
from pathlib import Path
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Nepali_Whisper_Project")

TRAIN_CSV = PROJECT_DIR / "train_manifest.csv"
VAL_CSV = PROJECT_DIR / "validation_manifest.csv"
TEST_CSV = PROJECT_DIR / "test_manifest.csv"

TRAIN_AUDIO_DIR = PROJECT_DIR / "audio" / "train"
VAL_AUDIO_DIR = PROJECT_DIR / "audio" / "validation"
TEST_AUDIO_DIR = PROJECT_DIR / "audio" / "test"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("===== ENVIRONMENT CHECK =====")
print("GPU:", torch.cuda.get_device_name(0))
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))
print("Training audio exists:", TRAIN_AUDIO_DIR.exists())
print("Validation audio exists:", VAL_AUDIO_DIR.exists())
print("Testing audio exists:", TEST_AUDIO_DIR.exists())

===== ENVIRONMENT CHECK =====
GPU: Tesla T4
Training samples: 34941
Validation samples: 7622
Testing samples: 7495
Training audio exists: True
Validation audio exists: True
Testing audio exists: True


In [6]:
from transformers import WhisperProcessor

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="nepali",
    task="transcribe"
)

processor.tokenizer.clean_up_tokenization_spaces = False

print("Whisper processor loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Whisper processor loaded successfully!


## 3. Prepare audio and transcripts

Whisper expects audio at 16 kHz. The preprocessing function opens each audio file, converts stereo audio to mono when needed, resamples it if necessary, creates Whisper input features, and tokenizes the reference transcript.

The dataset class applies this process when a sample is requested, rather than loading the whole audio dataset into memory at once.

In [7]:
import librosa
import soundfile as sf
import numpy as np

SAMPLING_RATE = 16000

def prepare_example(row, audio_dir):
    audio_path = audio_dir / f"{row['file_id']}.flac"

    audio, sr = sf.read(audio_path)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != SAMPLING_RATE:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=SAMPLING_RATE
        )

    input_features = processor(
        audio.astype(np.float32),
        sampling_rate=SAMPLING_RATE
    ).input_features[0]

    labels = processor.tokenizer(
        row["transcription"]
    ).input_ids

    return {
        "input_features": input_features,
        "labels": labels
    }

In [8]:
from torch.utils.data import Dataset

class WhisperDataset(Dataset):
    def __init__(self, dataframe, audio_dir):
        self.dataframe = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        return prepare_example(row, self.audio_dir)


train_dataset = WhisperDataset(train_df, TRAIN_AUDIO_DIR)
val_dataset = WhisperDataset(val_df, VAL_AUDIO_DIR)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 34941
Validation samples: 7622


## 4. Build batches for training

Audio features and transcript tokens can have different shapes across samples. The data collator combines samples into a batch.

Transcript padding is marked with `-100`, so those padding positions are ignored when the model calculates its loss. The decoder start token is removed when it is present at the beginning of every label sequence.

In [9]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch
import numpy as np

@dataclass
class WhisperDataCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]):
        input_features = torch.from_numpy(
            np.stack([item["input_features"] for item in features])
        ).float()

        label_features = [
            {"input_ids": item["labels"]}
            for item in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        return {
            "input_features": input_features,
            "labels": labels
        }


data_collator = WhisperDataCollator(
    processor=processor,
    decoder_start_token_id=50258
)

print("Data collator recreated successfully!")

Data collator recreated successfully!


## 5. Set the training configuration

The full training run uses a per-device batch size of 2 and gradient accumulation over 8 steps. This gives an effective batch size of 16.

FP16 and gradient checkpointing are enabled to reduce GPU memory use. Checkpoints are saved every 500 training steps so that training can be resumed if the Colab session disconnects.

In [10]:
from transformers import Seq2SeqTrainingArguments

FULL_OUTPUT_DIR = PROJECT_DIR / "whisper_small_fft" / "full_training"
FULL_CHECKPOINT_DIR = FULL_OUTPUT_DIR / "checkpoints"
RESULTS_DIR = FULL_OUTPUT_DIR / "results"

FULL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(FULL_CHECKPOINT_DIR),

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=1,

    fp16=True,
    gradient_checkpointing=True,

    learning_rate=1e-5,
    warmup_steps=100,
    weight_decay=0.01,

    eval_strategy="no",

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    logging_steps=25,
    report_to="none",

    seed=42,
    data_seed=42,

    dataloader_num_workers=2,
    predict_with_generate=False,

    save_only_model=False,
    remove_unused_columns=False
)

print("Training arguments recreated successfully!")

Training arguments recreated successfully!


## 6. Load the model and prepare the trainer

The pretrained `openai/whisper-small` model is loaded for full fine-tuning. Gradient checkpointing is enabled and the model cache is disabled during training to reduce memory use.

The trainer connects the model, training arguments, datasets, and data collator. The next cell checks whether a saved checkpoint is available before training continues.

In [11]:
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainer
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    attn_implementation="sdpa"
).to("cuda")

model.config.use_cache = False
model.gradient_checkpointing_enable()

model.generation_config.language = "nepali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor
)

print("Whisper model loaded!")
print("Trainer recreated!")
print("GPU:", torch.cuda.get_device_name(0))

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Whisper model loaded!
Trainer recreated!
GPU: Tesla T4


In [12]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(FULL_CHECKPOINT_DIR))

print("Latest checkpoint:", last_checkpoint)

if last_checkpoint:
    print("\nCheckpoint files:")
    print(os.listdir(last_checkpoint))
else:
    print("No checkpoint found!")

Latest checkpoint: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/checkpoints/checkpoint-1500

Checkpoint files:
['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


## 7. Resume and complete training

The training call resumes from the latest checkpoint found in the checkpoint directory. This was important for this experiment because training could not reliably be completed in one Colab session.

The saved output shows that training reached **2,184 global steps**, with a final reported training loss of approximately **1.0453**. The final model and processor are then saved to the project folder.

In [13]:
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

print("\n===== TRAINING COMPLETED =====")
print("Final global step:", trainer.state.global_step)
print("Training loss:", train_result.training_loss)

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss
1525,1.214208
1550,1.046878
1575,1.149947
1600,1.117202
1625,1.196227
1650,0.916141
1675,1.095275
1700,1.013800
1725,1.106165
1750,1.077792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== TRAINING COMPLETED =====
Final global step: 2184
Training loss: 1.0452976861195258


In [14]:
FINAL_MODEL_DIR = FULL_OUTPUT_DIR / "final_model"

trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

print("Final model saved successfully!")
print("Location:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved successfully!
Location: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/final_model


In [15]:
import os

print("Final model files:")
print(os.listdir(FINAL_MODEL_DIR))

print("\nModel directory exists:", FINAL_MODEL_DIR.exists())

Final model files:
['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']

Model directory exists: True


## 8. Run inference on the validation set

The saved fine-tuned model is loaded in evaluation mode. Validation audio is processed in batches, and the model generates a transcript for each sample.

Predictions are written to a CSV file as the batches are processed. Saving progress during inference makes it easier to inspect or recover the results if a runtime stops.

In [16]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

eval_processor = WhisperProcessor.from_pretrained(
    str(FINAL_MODEL_DIR)
)

eval_model = WhisperForConditionalGeneration.from_pretrained(
    str(FINAL_MODEL_DIR),
    attn_implementation="sdpa"
).to("cuda")

eval_model.eval()

print("Fine-tuned model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!
GPU: Tesla T4


In [17]:
eval_model.generation_config.language = "nepali"
eval_model.generation_config.task = "transcribe"
eval_model.generation_config.forced_decoder_ids = None

print("Language:", eval_model.generation_config.language)
print("Task:", eval_model.generation_config.task)
print("Validation samples:", len(val_df))

Language: nepali
Task: transcribe
Validation samples: 7622


In [18]:
import time
import pandas as pd
import numpy as np
import soundfile as sf
import librosa
from tqdm.auto import tqdm
from pathlib import Path

BATCH_SIZE = 8

validation_predictions = []
total_audio_duration = 0

start_time = time.time()

for start in tqdm(range(0, len(val_df), BATCH_SIZE)):

    batch_df = val_df.iloc[start:start + BATCH_SIZE]

    batch_features = []
    batch_references = []

    for _, row in batch_df.iterrows():

        audio_path = VAL_AUDIO_DIR / f"{row['file_id']}.flac"

        audio, sr = sf.read(audio_path)

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        if sr != 16000:
            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=16000
            )

        total_audio_duration += len(audio) / 16000

        features = eval_processor(
            audio.astype(np.float32),
            sampling_rate=16000
        ).input_features[0]

        batch_features.append(features)
        batch_references.append(row["transcription"])

    input_features = torch.tensor(
        np.stack(batch_features),
        dtype=torch.float32
    ).to("cuda")

    with torch.inference_mode():
        predicted_ids = eval_model.generate(
            input_features,
            language="nepali",
            task="transcribe"
        )

    batch_predictions = eval_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )

    for row, reference, prediction in zip(
        batch_df.itertuples(),
        batch_references,
        batch_predictions
    ):
        validation_predictions.append({
            "file_id": row.file_id,
            "reference": reference,
            "prediction": prediction
        })

    # Save progress after every batch
    pd.DataFrame(validation_predictions).to_csv(
        RESULTS_DIR / "validation_predictions.csv",
        index=False
    )

total_time = time.time() - start_time

print("\n===== VALIDATION INFERENCE COMPLETED =====")
print("Samples:", len(validation_predictions))
print("Total audio duration (hours):", total_audio_duration / 3600)
print("Inference time (minutes):", total_time / 60)
print("Average time per sample:", total_time / len(val_df))
print("Throughput:", len(val_df) / total_time)
print("RTF:", total_time / total_audio_duration)

  0%|          | 0/953 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



===== VALIDATION INFERENCE COMPLETED =====
Samples: 7622
Total audio duration (hours): 8.317194444444434
Inference time (minutes): 85.93094356060028
Average time per sample: 0.6764440584670712
Throughput: 1.4783188461528622
RTF: 0.1721953721586146


## 9. Evaluate validation predictions

Before calculating the scores, the prediction file is checked for missing values, duplicate IDs, and empty transcripts.

Text normalization is applied consistently to references and predictions. Word Error Rate (WER) measures word-level differences, while Character Error Rate (CER) measures character-level differences.

The following cells also inspect sample-level errors and look for repeated phrases in model predictions. These checks help identify cases that are not clear from the overall scores alone.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

# Project paths
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Nepali_Whisper_Project"
)

RESULTS_DIR = (
    PROJECT_DIR / "whisper_small_fft" / "full_training" / "results"
)

# Load validation predictions
csv_path = RESULTS_DIR / "validation_predictions.csv"

val_df = pd.read_csv(csv_path)

print("Total validation samples:", len(val_df))
print("\nColumns:", val_df.columns.tolist())
print("\nFirst 5 predictions:")
display(val_df.head())

Mounted at /content/drive
Total validation samples: 7622

Columns: ['file_id', 'reference', 'prediction']

First 5 predictions:


,file_id,reference,prediction
0,00f146a59a,नियमित बस उत्तर,नियमित ब सुत्तर
1,00a49d7c51,मेरीको विरुद्धमा भएको,मेरिको विरुद्धमा भएको
2,0061189667,करिब ३५ ४०,करिब ३५५
3,001907805e,दृष्टिकोणले किराँती विशेषता,दृष्टिकोणले किराती विशेषता
4,00a2b01b40,चरा कहाँ कस्तो,चरा कहाँ कस्तो


In [2]:
print("===== DATA QUALITY CHECK =====")

print("Total rows:", len(val_df))

print("\nMissing values:")
print(val_df.isnull().sum())

print("\nDuplicate file IDs:", val_df["file_id"].duplicated().sum())

print("\nEmpty references:",
      (val_df["reference"].fillna("").str.strip() == "").sum())

print("Empty predictions:",
      (val_df["prediction"].fillna("").str.strip() == "").sum())

print("\nReference word count:")
print(val_df["reference"].str.split().str.len().describe())

print("\nPrediction word count:")
print(val_df["prediction"].str.split().str.len().describe())

===== DATA QUALITY CHECK =====
Total rows: 7622

Missing values:
file_id       0
reference     0
prediction    0
dtype: int64

Duplicate file IDs: 0

Empty references: 0
Empty predictions: 0

Reference word count:
count    7622.000000
mean        2.984125
std         1.072927
min         1.000000
25%         3.000000
50%         3.000000
75%         3.000000
max        15.000000
Name: reference, dtype: float64

Prediction word count:
count    7622.000000
mean        2.977171
std         1.008430
min         1.000000
25%         3.000000
50%         3.000000
75%         3.000000
max        15.000000
Name: prediction, dtype: float64


In [5]:
!pip install -q jiwer

In [6]:
import re
import unicodedata
import jiwer
from jiwer import wer, cer

# Same normalization used for baseline
def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))

    # Remove punctuation
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply normalization
val_df["ref_clean"] = val_df["reference"].apply(normalize_text)
val_df["pred_clean"] = val_df["prediction"].apply(normalize_text)

# Corpus-level evaluation
references = val_df["ref_clean"].tolist()
predictions = val_df["pred_clean"].tolist()

validation_wer = wer(references, predictions)
validation_cer = cer(references, predictions)

print("===== VALIDATION RESULTS =====")
print(f"Validation WER: {validation_wer:.4f} ({validation_wer*100:.2f}%)")
print(f"Validation CER: {validation_cer:.4f} ({validation_cer*100:.2f}%)")

===== VALIDATION RESULTS =====
Validation WER: 0.2885 (28.85%)
Validation CER: 0.0711 (7.11%)


In [7]:
from jiwer import wer

# Calculate WER for each individual sample
val_df["sample_wer"] = [
    wer(ref, pred)
    for ref, pred in zip(
        val_df["ref_clean"],
        val_df["pred_clean"]
    )
]

print("===== SAMPLE-LEVEL ERROR ANALYSIS =====")

print("Exact matches:",
      (val_df["sample_wer"] == 0).sum())

print("Samples with errors:",
      (val_df["sample_wer"] > 0).sum())

print("\nSample WER distribution:")
print(val_df["sample_wer"].describe())

print("\nError thresholds:")
for threshold in [1, 2, 3, 5, 10]:
    count = (val_df["sample_wer"] >= threshold).sum()
    percentage = count / len(val_df) * 100

    print(
        f"WER >= {threshold}: "
        f"{count} samples ({percentage:.2f}%)"
    )

print("\n===== 10 WORST PREDICTIONS =====")

worst_samples = val_df.nlargest(10, "sample_wer")

display(
    worst_samples[
        ["file_id", "ref_clean", "pred_clean", "sample_wer"]
    ]
)

===== SAMPLE-LEVEL ERROR ANALYSIS =====
Exact matches: 3555
Samples with errors: 4067

Sample WER distribution:
count    7622.000000
mean        0.290628
std         0.335228
min         0.000000
25%         0.000000
50%         0.333333
75%         0.500000
max         3.000000
Name: sample_wer, dtype: float64

Error thresholds:
WER >= 1: 595 samples (7.81%)
WER >= 2: 9 samples (0.12%)
WER >= 3: 1 samples (0.01%)
WER >= 5: 0 samples (0.00%)
WER >= 10: 0 samples (0.00%)

===== 10 WORST PREDICTIONS =====


,file_id,ref_clean,pred_clean,sample_wer
7056,ece6ffcdbf,कालीगण्डकी,काली गण्ड किष,3.000000
201,06d84bed7b,कालीबंगा,काली बङ्गा,2.000000
774,1999609210,मोडिन्छ,मोडिन छ,2.000000
2956,6230ced5b6,कालोटाउके विरहीचरी,कालो टाउके विरही चरी,2.000000
3375,702efcaaad,खेल्छन्,खेल छन्,2.000000
3692,7a55970de0,केपटाउनबाट,केपटाउँ बाट,2.000000
5939,c72da180e8,लोलीगाउँ,लोलि गाउँ,2.000000
6924,e7b515d705,मिसिन्छन्,मिसिन छन्,2.000000
7588,ff41299c9c,सबैजना रमाउँछन्,सबै जाना र माओँछन्,2.000000
424,0d77f20229,जल्दापारा वन्यजीव अभयारण्य,जल्दा पारा भनिजीव अभएयार यौँ,1.666667


In [8]:
from collections import Counter

def count_repeated_ngrams(text, n):
    words = text.split()

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words)-n+1)
    ]

    counts = Counter(ngrams)

    return sum(count - 1 for count in counts.values() if count > 1)


for n in [2, 3, 4]:
    val_df[f"repeat_{n}gram"] = (
        val_df["pred_clean"]
        .apply(lambda x: count_repeated_ngrams(x, n))
    )

val_df["repetition_score"] = (
    val_df["repeat_2gram"]
    + val_df["repeat_3gram"]
    + val_df["repeat_4gram"]
)

repeated_samples = val_df[
    val_df["repetition_score"] > 0
]

print("===== REPETITION ANALYSIS =====")

print("Samples with repetition:", len(repeated_samples))

print("\nRepetition by n-gram:")
print("2-gram:", (val_df["repeat_2gram"] > 0).sum())
print("3-gram:", (val_df["repeat_3gram"] > 0).sum())
print("4-gram:", (val_df["repeat_4gram"] > 0).sum())

print("\nTop repeated predictions:")

display(
    repeated_samples.nlargest(10, "repetition_score")[
        [
            "file_id",
            "ref_clean",
            "pred_clean",
            "repeat_2gram",
            "repeat_3gram",
            "repeat_4gram",
            "repetition_score"
        ]
    ]
)

===== REPETITION ANALYSIS =====
Samples with repetition: 1

Repetition by n-gram:
2-gram: 1
3-gram: 0
4-gram: 0

Top repeated predictions:


,file_id,ref_clean,pred_clean,repeat_2gram,repeat_3gram,repeat_4gram,repetition_score
3272,6c12f59f87,नगरकोट काठमाडौँबाट मात्र ३२ किलोमिटरको दूरिमा ...,नगरकोट काठमाडौँबाट मात्र ३२ किलोमिटरको दुरीमा ...,1,0,0,1


In [9]:
import json

# Save detailed validation predictions and sample-level errors
validation_csv = RESULTS_DIR / "validation_evaluation.csv"
val_df.to_csv(validation_csv, index=False)

# Save summary metrics
validation_summary = {
    "model": "Whisper Small - Full Fine-Tuning",
    "split": "validation",
    "samples": len(val_df),
    "corpus_wer": validation_wer,
    "corpus_cer": validation_cer,
    "exact_matches": int((val_df["sample_wer"] == 0).sum()),
    "exact_match_rate": float(
        (val_df["sample_wer"] == 0).mean()
    ),
    "samples_with_errors": int(
        (val_df["sample_wer"] > 0).sum()
    ),
    "mean_sample_wer": float(val_df["sample_wer"].mean()),
    "median_sample_wer": float(val_df["sample_wer"].median()),
    "repetition_samples": len(repeated_samples)
}

summary_path = RESULTS_DIR / "validation_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=4, ensure_ascii=False)

print("Validation evaluation saved!")
print(validation_csv)
print(summary_path)

Validation evaluation saved!
/content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/results/validation_evaluation.csv
/content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/results/validation_summary.json


## 10. Evaluate the test set

The test split is loaded separately from the training and validation splits. The fine-tuned model generates predictions for all test audio files, and the predictions are saved to a CSV file.

The test evaluation calculates WER and CER, counts exact matches, and displays examples with larger errors. The test set is used here for the final evaluation rather than for adjusting the model.

In [10]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Nepali_Whisper_Project"
)

TEST_AUDIO_DIR = PROJECT_DIR / "audio" / "test"
TEST_MANIFEST = PROJECT_DIR / "test_manifest.csv"

FINAL_MODEL_DIR = (
    PROJECT_DIR
    / "whisper_small_fft"
    / "full_training"
    / "final_model"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "whisper_small_fft"
    / "full_training"
    / "results"
)

test_df = pd.read_csv(TEST_MANIFEST)

print("===== TEST SET CHECK =====")
print("Test samples:", len(test_df))
print("Test audio folder exists:", TEST_AUDIO_DIR.exists())
print("Final model exists:", FINAL_MODEL_DIR.exists())
print("Results folder exists:", RESULTS_DIR.exists())

print("\nTest manifest columns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows:")
display(test_df.head())

print("\nExisting result files:")
for file in RESULTS_DIR.iterdir():
    print(file.name)

===== TEST SET CHECK =====
Test samples: 7495
Test audio folder exists: True
Final model exists: True
Results folder exists: True

Test manifest columns:
['file_id', 'archive_prefix', 'archive_path', 'file_size', 'compressed_size', 'speaker_id', 'transcription', 'split']

First 5 rows:


,file_id,archive_prefix,archive_path,file_size,compressed_size,speaker_id,transcription,split
0,00ee752d8d,0,asr_nepali/data/00/00ee752d8d.flac,51006,51006,e72b0,गेटा आँखा अस्पताल,test
1,006bd9a275,0,asr_nepali/data/00/006bd9a275.flac,59387,59387,6d3c6,प्रस्तोता र भारतीय,test
2,007e72e734,0,asr_nepali/data/00/007e72e734.flac,44974,44974,d0cd7,छुट्टाउँदै कर्णाली चिसापानी,test
3,00dbbdaa2c,0,asr_nepali/data/00/00dbbdaa2c.flac,53299,53299,d8cef,भएको हुन्छ,test
4,00b316fa0b,0,asr_nepali/data/00/00b316fa0b.flac,74223,74223,5f3e5,अन्यले राष्ट्रिय,test



Existing result files:
validation_predictions.csv
validation_evaluation.csv
validation_summary.json


In [11]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load processor
processor = WhisperProcessor.from_pretrained(
    str(FINAL_MODEL_DIR)
)

# Load fine-tuned model
model = WhisperForConditionalGeneration.from_pretrained(
    str(FINAL_MODEL_DIR),
    attn_implementation="sdpa"
).to("cuda")

model.eval()

# Configure Nepali transcription
model.generation_config.language = "nepali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print("Fine-tuned model loaded successfully!")
print("Device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!
Device: cuda:0
GPU: Tesla T4


In [12]:
import soundfile as sf

# Check first 5 test audio files
for file_id in test_df["file_id"].head(5):
    audio_path = TEST_AUDIO_DIR / f"{file_id}.flac"

    print(
        file_id,
        "| Exists:", audio_path.exists(),
        "| Path:", audio_path
    )

00ee752d8d | Exists: True | Path: /content/drive/MyDrive/Nepali_Whisper_Project/audio/test/00ee752d8d.flac
006bd9a275 | Exists: True | Path: /content/drive/MyDrive/Nepali_Whisper_Project/audio/test/006bd9a275.flac
007e72e734 | Exists: True | Path: /content/drive/MyDrive/Nepali_Whisper_Project/audio/test/007e72e734.flac
00dbbdaa2c | Exists: True | Path: /content/drive/MyDrive/Nepali_Whisper_Project/audio/test/00dbbdaa2c.flac
00b316fa0b | Exists: True | Path: /content/drive/MyDrive/Nepali_Whisper_Project/audio/test/00b316fa0b.flac


In [13]:
import time
import soundfile as sf
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pathlib import Path

TEST_PREDICTIONS_PATH = RESULTS_DIR / "test_predictions.csv"

BATCH_SIZE = 8

test_predictions = []
total_audio_seconds = 0

# Start timer
start_time = time.time()

for start_idx in tqdm(
    range(0, len(test_df), BATCH_SIZE),
    desc="Test inference"
):
    batch_df = test_df.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_audio = []
    batch_references = []
    batch_file_ids = []

    for _, row in batch_df.iterrows():

        file_id = row["file_id"]
        audio_path = TEST_AUDIO_DIR / f"{file_id}.flac"

        audio, sampling_rate = sf.read(audio_path)

        # Convert stereo to mono if necessary
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        # Resample if necessary
        if sampling_rate != 16000:
            import librosa

            audio = librosa.resample(
                audio,
                orig_sr=sampling_rate,
                target_sr=16000
            )

        audio = audio.astype(np.float32)

        total_audio_seconds += len(audio) / 16000

        batch_audio.append(audio)
        batch_references.append(row["transcription"])
        batch_file_ids.append(file_id)

    # Extract log-mel features
    inputs = processor(
        batch_audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding="max_length",
        truncation=True
    )

    input_features = inputs.input_features.to("cuda")

    # Generate predictions
    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features,
            language="nepali",
            task="transcribe"
        )

    batch_decoded = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )

    # Store predictions
    for file_id, reference, prediction in zip(
        batch_file_ids,
        batch_references,
        batch_decoded
    ):
        test_predictions.append({
            "file_id": file_id,
            "reference": reference,
            "prediction": prediction
        })

    # Save progress after every batch
    pd.DataFrame(test_predictions).to_csv(
        TEST_PREDICTIONS_PATH,
        index=False
    )

# End timer
end_time = time.time()

inference_seconds = end_time - start_time

print("\n===== TEST INFERENCE COMPLETED =====")
print("Samples:", len(test_predictions))
print("Total audio duration (hours):",
      total_audio_seconds / 3600)
print("Inference time (minutes):",
      inference_seconds / 60)
print("Average time per sample:",
      inference_seconds / len(test_predictions))
print("Throughput:",
      len(test_predictions) / inference_seconds)
print("RTF:",
      inference_seconds / total_audio_seconds)

print("\nPredictions saved to:", TEST_PREDICTIONS_PATH)

Test inference:   0%|          | 0/937 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



===== TEST INFERENCE COMPLETED =====
Samples: 7495
Total audio duration (hours): 7.8641666666666685
Inference time (minutes): 83.26429612636566
Average time per sample: 0.6665587415052622
Throughput: 1.5002428709309867
RTF: 0.1764634865452276

Predictions saved to: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/results/test_predictions.csv


In [14]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

TEST_PREDICTIONS_PATH = RESULTS_DIR / "test_predictions.csv"

test_results = pd.read_csv(TEST_PREDICTIONS_PATH)

print("Test samples:", len(test_results))
print("Missing values:")
print(test_results.isnull().sum())


# Same normalization used in previous evaluations
def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))

    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


test_results["ref_clean"] = (
    test_results["reference"].apply(normalize_text)
)

test_results["pred_clean"] = (
    test_results["prediction"].apply(normalize_text)
)


# Corpus-level metrics
test_references = test_results["ref_clean"].tolist()
test_predictions = test_results["pred_clean"].tolist()

test_wer = wer(test_references, test_predictions)
test_cer = cer(test_references, test_predictions)

print("\n===== FINE-TUNED MODEL: TEST RESULTS =====")
print(f"Test WER: {test_wer:.4f} ({test_wer*100:.2f}%)")
print(f"Test CER: {test_cer:.4f} ({test_cer*100:.2f}%)")

Test samples: 7495
Missing values:
file_id       0
reference     0
prediction    0
dtype: int64

===== FINE-TUNED MODEL: TEST RESULTS =====
Test WER: 0.2803 (28.03%)
Test CER: 0.0678 (6.78%)


In [15]:
from jiwer import wer

# Individual sample WER
test_results["sample_wer"] = [
    wer(ref, pred)
    for ref, pred in zip(
        test_results["ref_clean"],
        test_results["pred_clean"]
    )
]

print("===== TEST SAMPLE-LEVEL ANALYSIS =====")

print("Exact matches:",
      (test_results["sample_wer"] == 0).sum())

print("Samples with errors:",
      (test_results["sample_wer"] > 0).sum())

print("\nSample WER distribution:")
print(test_results["sample_wer"].describe())

print("\nError thresholds:")

for threshold in [1, 2, 3, 5, 10]:
    count = (
        test_results["sample_wer"] >= threshold
    ).sum()

    percentage = count / len(test_results) * 100

    print(
        f"WER >= {threshold}: "
        f"{count} samples ({percentage:.2f}%)"
    )

print("\n===== 10 WORST PREDICTIONS =====")

display(
    test_results.nlargest(10, "sample_wer")[
        ["file_id", "ref_clean", "pred_clean", "sample_wer"]
    ]
)

===== TEST SAMPLE-LEVEL ANALYSIS =====
Exact matches: 3619
Samples with errors: 3876

Sample WER distribution:
count    7495.000000
mean        0.280178
std         0.331189
min         0.000000
25%         0.000000
50%         0.250000
75%         0.500000
max         3.000000
Name: sample_wer, dtype: float64

Error thresholds:
WER >= 1: 562 samples (7.50%)
WER >= 2: 8 samples (0.11%)
WER >= 3: 1 samples (0.01%)
WER >= 5: 0 samples (0.00%)
WER >= 10: 0 samples (0.00%)

===== 10 WORST PREDICTIONS =====


,file_id,ref_clean,pred_clean,sample_wer
6294,d829821c0c,जलपाइगुड़ीदेखि,जल पाइ गुडीदेखि,3.00
3074,6809589bc9,ज्यामितिमा,जमी तीमा,2.00
3316,7029cee6ae,मन्त्रीमण्डलमा,मन्त्री मन्दलमा,2.00
3822,8331e6e0ae,कालीमुन्टेलाई,काली मुन्टेलाई,2.00
4279,92fb639f43,अनुवर्णसूची अनुवर्गसूची,अनुवर्ण सूची अनुवर्ग सूची,2.00
5021,adc61cb582,कालेगाँडा,कालै गाँडा,2.00
5234,b45f2410d5,तर टर्‍यो,त र टर यो,2.00
6053,cffdd4a632,सिन्धुपाल्चोकको संस्थापकमध्ये,सिन्धु बाल्जोको संस्थापक मध्ये,2.00
124,0496cd16ce,१७०००० सेना रहेको देखियो,एक साथ सुन्य सुन्य सुन्य सेनारहेको देख्यो,1.75
159,0517bc6a66,जिएनयुलिनक्समा आधारित,जिएन युनि नक्सामा आधारित,1.50


In [16]:
from collections import Counter

def count_repeated_ngrams(text, n):
    words = str(text).split()

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

    counts = Counter(ngrams)

    return sum(count - 1 for count in counts.values() if count > 1)


# Detect repeated n-grams
for n in [2, 3, 4]:
    test_results[f"repeat_{n}gram"] = (
        test_results["pred_clean"].apply(
            lambda text: count_repeated_ngrams(text, n)
        )
    )


# Combined repetition score
test_results["repetition_score"] = (
    test_results["repeat_2gram"]
    + test_results["repeat_3gram"]
    + test_results["repeat_4gram"]
)

repeated_samples = test_results[
    test_results["repetition_score"] > 0
]

print("===== TEST REPETITION ANALYSIS =====")

print("Repetition-like samples:", len(repeated_samples))

for n in [2, 3, 4]:
    count = (test_results[f"repeat_{n}gram"] > 0).sum()
    print(f"Samples with repeated {n}-grams: {count}")

print("\n===== TOP 10 REPETITION SAMPLES =====")

display(
    repeated_samples.nlargest(10, "repetition_score")[
        [
            "file_id",
            "ref_clean",
            "pred_clean",
            "repeat_2gram",
            "repeat_3gram",
            "repeat_4gram",
            "repetition_score"
        ]
    ]
)

# Save complete test evaluation
TEST_EVALUATION_PATH = RESULTS_DIR / "test_evaluation.csv"
test_results.to_csv(TEST_EVALUATION_PATH, index=False)

print("\nSaved:", TEST_EVALUATION_PATH)

===== TEST REPETITION ANALYSIS =====
Repetition-like samples: 1
Samples with repeated 2-grams: 1
Samples with repeated 3-grams: 0
Samples with repeated 4-grams: 0

===== TOP 10 REPETITION SAMPLES =====


,file_id,ref_clean,pred_clean,repeat_2gram,repeat_3gram,repeat_4gram,repetition_score
124,0496cd16ce,१७०००० सेना रहेको देखियो,एक साथ सुन्य सुन्य सुन्य सेनारहेको देख्यो,1,0,0,1



Saved: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/results/test_evaluation.csv


## 11. Compare with pretrained Whisper Small

The baseline predictions were generated using the original pretrained Whisper Small model. They are loaded and compared with the fine-tuned model's predictions.

Before comparing the scores, the file IDs and reference transcripts are checked to make sure both models were evaluated on the same test samples. The paired comparison then counts how many individual samples improved, worsened, or stayed the same.

The summary and paired results are saved for later use.

In [21]:
baseline_path = PROJECT_DIR / "results" / "zero_shot_whisper_small_predictions.csv"

baseline_results = pd.read_csv(baseline_path)

print("Baseline samples:", len(baseline_results))
print("\nColumns:")
print(baseline_results.columns.tolist())

display(baseline_results.head())

Baseline samples: 7495

Columns:
['file_id', 'reference', 'prediction', 'duration_seconds']


,file_id,reference,prediction,duration_seconds
0,00ee752d8d,गेटा आँखा अस्पताल,गेटा आखा अस्पताल,3.4
1,006bd9a275,प्रस्तोता र भारतीय,प्रस्टो तार बारतिय,3.4
2,007e72e734,छुट्टाउँदै कर्णाली चिसापानी,जद्ताउने करनाली ची सापानी,2.3
3,00dbbdaa2c,भएको हुन्छ,बाए को ओन्चा,3.0
4,00b316fa0b,अन्यले राष्ट्रिय,अन्नेले रास्टिया,3.8


In [22]:
from jiwer import wer, cer

# Apply the same normalization
baseline_results["ref_clean"] = (
    baseline_results["reference"].apply(normalize_text)
)
baseline_results["pred_clean"] = (
    baseline_results["prediction"].apply(normalize_text)
)

# Verify test-set alignment
baseline_check = baseline_results[
    ["file_id", "ref_clean"]
].merge(
    test_results[["file_id", "ref_clean"]],
    on="file_id",
    suffixes=("_baseline", "_finetuned"),
    how="outer",
    indicator=True
)

print("===== DATA ALIGNMENT CHECK =====")
print("Matching file IDs:",
      (baseline_check["_merge"] == "both").sum())

print("Missing/mismatched IDs:",
      (baseline_check["_merge"] != "both").sum())

print("Reference mismatches:",
      (baseline_check["ref_clean_baseline"] !=
       baseline_check["ref_clean_finetuned"]).sum())

# Calculate baseline metrics
baseline_wer = wer(
    baseline_results["ref_clean"].tolist(),
    baseline_results["pred_clean"].tolist()
)

baseline_cer = cer(
    baseline_results["ref_clean"].tolist(),
    baseline_results["pred_clean"].tolist()
)

print("\n===== PRETRAINED WHISPER SMALL =====")
print(f"WER: {baseline_wer:.4f} ({baseline_wer*100:.2f}%)")
print(f"CER: {baseline_cer:.4f} ({baseline_cer*100:.2f}%)")

print("\n===== FINE-TUNED WHISPER SMALL =====")
print(f"WER: {test_wer:.4f} ({test_wer*100:.2f}%)")
print(f"CER: {test_cer:.4f} ({test_cer*100:.2f}%)")

print("\n===== RELATIVE ERROR REDUCTION =====")
print(
    f"WER reduction: "
    f"{(baseline_wer-test_wer)/baseline_wer*100:.2f}%"
)

print(
    f"CER reduction: "
    f"{(baseline_cer-test_cer)/baseline_cer*100:.2f}%"
)

===== DATA ALIGNMENT CHECK =====
Matching file IDs: 7495
Missing/mismatched IDs: 0
Reference mismatches: 0

===== PRETRAINED WHISPER SMALL =====
WER: 3.2321 (323.21%)
CER: 1.8695 (186.95%)

===== FINE-TUNED WHISPER SMALL =====
WER: 0.2803 (28.03%)
CER: 0.0678 (6.78%)

===== RELATIVE ERROR REDUCTION =====
WER reduction: 91.33%
CER reduction: 96.37%


In [23]:
# Calculate sample-level WER for pretrained baseline
baseline_results["sample_wer"] = [
    wer(ref, pred)
    for ref, pred in zip(
        baseline_results["ref_clean"],
        baseline_results["pred_clean"]
    )
]

# Align both models by file_id
paired = baseline_results[
    ["file_id", "sample_wer"]
].merge(
    test_results[["file_id", "sample_wer"]],
    on="file_id",
    suffixes=("_baseline", "_finetuned")
)

# Compare individual samples
paired["wer_change"] = (
    paired["sample_wer_baseline"]
    - paired["sample_wer_finetuned"]
)

improved = (paired["wer_change"] > 0).sum()
worsened = (paired["wer_change"] < 0).sum()
unchanged = (paired["wer_change"] == 0).sum()

print("===== PAIRED SAMPLE COMPARISON =====")
print("Improved:", improved)
print("Worsened:", worsened)
print("Unchanged:", unchanged)

print("\nPercentages:")
print(f"Improved: {improved/len(paired)*100:.2f}%")
print(f"Worsened: {worsened/len(paired)*100:.2f}%")
print(f"Unchanged: {unchanged/len(paired)*100:.2f}%")

print("\n===== EXACT MATCH COMPARISON =====")
print(
    "Baseline exact matches:",
    (baseline_results["sample_wer"] == 0).sum()
)

print(
    "Fine-tuned exact matches:",
    (test_results["sample_wer"] == 0).sum()
)

===== PAIRED SAMPLE COMPARISON =====
Improved: 7136
Worsened: 41
Unchanged: 318

Percentages:
Improved: 95.21%
Worsened: 0.55%
Unchanged: 4.24%

===== EXACT MATCH COMPARISON =====
Baseline exact matches: 30
Fine-tuned exact matches: 3619


In [24]:
import json

benchmark_summary = {
    "model": "Whisper Small - Full Fine-Tuning",
    "test_samples": len(paired),

    "pretrained": {
        "wer": baseline_wer,
        "cer": baseline_cer,
        "exact_matches": int(
            (baseline_results["sample_wer"] == 0).sum()
        ),
        "inference_minutes": 90.23
    },

    "full_finetuning": {
        "wer": test_wer,
        "cer": test_cer,
        "exact_matches": int(
            (test_results["sample_wer"] == 0).sum()
        ),
        "inference_minutes": 83.2643
    },

    "relative_error_reduction": {
        "wer_percent": (
            (baseline_wer - test_wer) / baseline_wer * 100
        ),
        "cer_percent": (
            (baseline_cer - test_cer) / baseline_cer * 100
        )
    },

    "paired_comparison": {
        "improved": int(improved),
        "worsened": int(worsened),
        "unchanged": int(unchanged)
    }
}

summary_path = RESULTS_DIR / "fft_benchmark_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_summary, f, indent=4, ensure_ascii=False)

paired.to_csv(
    RESULTS_DIR / "fft_paired_comparison.csv",
    index=False
)

print("Benchmark summary saved:", summary_path)
print("Paired comparison saved.")

Benchmark summary saved: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/results/fft_benchmark_summary.json
Paired comparison saved.


## 12. Main results from this run

The saved evaluation outputs report the following test-set results:

- Pretrained Whisper Small: WER **323.21%**, CER **186.95%**
- Fine-tuned Whisper Small: WER **28.03%**, CER **6.78%**
- Relative error reduction: WER **91.33%**, CER **96.37%**
- Of the 7,495 paired test samples, 7,136 improved, 41 worsened, and 318 stayed the same.

These values are copied from the notebook's existing outputs. They should be interpreted in the context of the dataset and the normalization used in this experiment.

## 13. Result and project file checks

These cells list the files created during the experiment and the files stored in the project directory. They are simple checks to confirm where predictions, evaluation files, and other project data were saved.

In [17]:
import os

print("Files in results directory:\n")

for file in os.listdir(RESULTS_DIR):
    print(file)

Files in results directory:

validation_predictions.csv
validation_evaluation.csv
validation_summary.json
test_predictions.csv
test_evaluation.csv


In [19]:
print("Project root:", PROJECT_DIR)

print("\nFiles in project root:")
for path in PROJECT_DIR.iterdir():
    print(path.name)

Project root: /content/drive/MyDrive/Nepali_Whisper_Project

Files in project root:
metadata.csv
audio_manifest.csv
duration_sample.csv
duration_statistics.csv
speaker_statistics.csv
speaker_assignment.csv
train_manifest.csv
validation_manifest.csv
test_manifest.csv
selected_manifest.csv
audio
audio_quality_report_backup.csv
audio_quality_report.csv
transcription_quality_report.csv
transcription_quality_report_backup.csv
download_progress.csv
drive_sync_test.txt
results
whisper_small_fft


In [20]:
ROOT_RESULTS_DIR = PROJECT_DIR / "results"

for path in ROOT_RESULTS_DIR.iterdir():
    print(path.name)

zero_shot_checkpoint_bs8.csv
zero_shot_whisper_small_predictions.csv



## Appendix. Additional checks and short pilot run

These cells are kept because they record checks that were useful during development but are not part of the final training and evaluation pipeline:

- inspecting one preprocessed audio sample;
- checking the shape of a training batch and its padding;
- checking that a forward and backward pass work on the GPU;
- running a short pilot training session;
- checking the pilot checkpoints and testing whether training can resume.

The repeated model loading, preprocessing, data collator, training-argument, and trainer setup cells have been removed because those steps are already documented in the main workflow. The later recovery section was also removed because it repeated the same setup and full-training steps.


In [ ]:
sample = train_data.iloc[0]

processed_sample = prepare_example(
    sample,
    TRAIN_AUDIO_DIR
)

print("===== PREPROCESSING TEST =====")
print("File ID:", sample["file_id"])
print("Reference transcript:", sample["transcription"])

print("\nInput feature shape:",
      processed_sample["input_features"].shape)

print("Label token count:",
      len(processed_sample["labels"]))

print("Decoded labels:",
      processor.tokenizer.decode(processed_sample["labels"]))

===== PREPROCESSING TEST =====
File ID: 003017eaca
Reference transcript: थिए तर पछि

Input feature shape: (80, 3000)
Label token count: 18
Decoded labels: <|startoftranscript|><|ne|><|transcribe|><|notimestamps|>थिए तर पछि<|endoftext|>


In [ ]:
# ============================================
# 7. TEST TRAINING BATCH
# ============================================

from torch.utils.data import DataLoader

test_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

batch = next(iter(test_loader))

print("===== BATCH TEST =====")
print("Input features:", batch["input_features"].shape)
print("Labels:", batch["labels"].shape)

print("\nLabel values:")
print(batch["labels"][0][:30])

# Check padding
print("\nNumber of ignored padding tokens:",
      (batch["labels"] == -100).sum().item())

# Check GPU memory
torch.cuda.empty_cache()

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

===== BATCH TEST =====
Input features: torch.Size([2, 80, 3000])
Labels: torch.Size([2, 23])

Label values:
tensor([50313, 50359, 50363,  3941,    98, 33279,  3941,   237,  8485,    97,
        25411,  8485,   103,  3941,   249, 33279, 50257,  -100,  -100,  -100,
         -100,  -100,  -100])

Number of ignored padding tokens: 6

GPU memory allocated: 0.47 GB
GPU memory reserved: 0.49 GB


In [ ]:

# Prepare one sample
single_sample = data_collator([train_dataset[0]])

# Move batch to GPU
single_sample = {
    key: value.to("cuda")
    for key, value in single_sample.items()
}

model.train()
model.zero_grad(set_to_none=True)

torch.cuda.reset_peak_memory_stats()

# Forward pass
with torch.autocast("cuda", dtype=torch.float16):
    outputs = model(**single_sample)
    loss = outputs.loss

print("Training loss:", loss.item())

# Backward pass
loss.backward()

print("Backward pass successful!")

# Check gradients
grad_exists = any(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Gradients calculated:", grad_exists)

print("Peak GPU memory:",
      round(torch.cuda.max_memory_allocated() / 1024**3, 2),
      "GB")

# Clear test gradients
model.zero_grad(set_to_none=True)
model.eval()

Training loss: 2.295090675354004
Backward pass successful!
Gradients calculated: True
Peak GPU memory: 1.56 GB


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
 

In [ ]:
# ============================================
# 13. PILOT TRAINING
# ============================================

# Temporarily configure a 10-step pilot
trainer.args.max_steps = 10
trainer.args.save_steps = 5
trainer.args.logging_steps = 1

print("Starting 10-step pilot...")

# Start training
pilot_result = trainer.train()

print("\n===== PILOT RESULTS =====")
print("Training loss:", pilot_result.training_loss)
print("Training steps:", pilot_result.global_step)

print("\nSaved checkpoints:")
for checkpoint in sorted(CHECKPOINT_DIR.glob("checkpoint-*")):
    print(checkpoint.name)

Starting 10-step pilot...


Step,Training Loss,Validation Loss
10,12.606188,1.650143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== PILOT RESULTS =====
Training loss: 13.787283897399902
Training steps: 10

Saved checkpoints:
checkpoint-10
checkpoint-5


In [ ]:
# ============================================
# 14. VERIFY CHECKPOINT CONTENTS
# ============================================

for step in [5, 10]:

    checkpoint_path = CHECKPOINT_DIR / f"checkpoint-{step}"

    print(f"\n===== CHECKPOINT {step} =====")

    if checkpoint_path.exists():

        files = sorted(os.listdir(checkpoint_path))

        for file in files:
            print("✓", file)

        required_files = [
            "optimizer.pt",
            "scheduler.pt",
            "trainer_state.json",
            "training_args.bin"
        ]

        print("\nResume components:")
        for file in required_files:
            print(file, ":", file in files)

    else:
        print("Checkpoint not found!")


===== CHECKPOINT 5 =====
✓ config.json
✓ generation_config.json
✓ model.safetensors
✓ optimizer.pt
✓ preprocessor_config.json
✓ rng_state.pth
✓ scaler.pt
✓ scheduler.pt
✓ trainer_state.json
✓ training_args.bin

Resume components:
optimizer.pt : True
scheduler.pt : True
trainer_state.json : True
training_args.bin : True

===== CHECKPOINT 10 =====
✓ config.json
✓ generation_config.json
✓ model.safetensors
✓ optimizer.pt
✓ preprocessor_config.json
✓ rng_state.pth
✓ scaler.pt
✓ scheduler.pt
✓ trainer_state.json
✓ training_args.bin

Resume components:
optimizer.pt : True
scheduler.pt : True
trainer_state.json : True
training_args.bin : True


In [ ]:
# ============================================
# 15. TEST CHECKPOINT RESUMPTION
# ============================================

resume_checkpoint = CHECKPOINT_DIR / "checkpoint-5"

# Set a short test target
trainer.args.max_steps = 7
trainer.args.save_steps = 2
trainer.args.logging_steps = 1

print("Resuming from:", resume_checkpoint)
print("Expected final global step: 7")

resume_result = trainer.train(
    resume_from_checkpoint=str(resume_checkpoint)
)

print("\n===== RESUME TEST RESULTS =====")
print("Final global step:", trainer.state.global_step)
print("Training loss:", resume_result.training_loss)

Resuming from: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/checkpoints/checkpoint-5
Expected final global step: 7


[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	save_steps: 2 (from args) != 5 (from trainer_state.json)


Step,Training Loss,Validation Loss
7,14.784839,1.720138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== RESUME TEST RESULTS =====
Final global step: 7
Training loss: 14.485911846160889
